In [6]:
# ─────────────────────────────────────────────────────────────
# ISL Non-Manual Feature Real-Time Webcam Inference
# ONNX Runtime + OpenCV + MediaPipe
# ─────────────────────────────────────────────────────────────

import cv2
import numpy as np
import onnxruntime as ort
from collections import deque, Counter
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────

SEQ_LEN = 30

EXPR_CLASSES = [
    'neutral',
    'happy',
    'sad',
    'surprise',
    'fear',
    'disgust',
    'anger',
    'contempt'
]

HEAD_CLASSES = [
    'nod',
    'shake',
    'tilt',
    'still'
]

# ─────────────────────────────────────────────────────────────
# LOAD ONNX MODELS
# ─────────────────────────────────────────────────────────────

expr_session = ort.InferenceSession(
    'isl_nmf/onnx/expression_cnn.onnx',
    providers=['CPUExecutionProvider']
)

hm_session = ort.InferenceSession(
    'isl_nmf/onnx/head_movement_tcn.onnx',
    providers=['CPUExecutionProvider']
)

print('✓ ONNX models loaded')

# ─────────────────────────────────────────────────────────────
# MEDIAPIPE FACE LANDMARKER
# ─────────────────────────────────────────────────────────────

model_path = 'face_landmarker.task'

base_options = python.BaseOptions(model_asset_path=model_path)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    #min_face_presence_score=0.5,
    min_tracking_confidence=0.5
)

face_landmarker = vision.FaceLandmarker.create_from_options(options)

print('✓ MediaPipe loaded')

# ─────────────────────────────────────────────────────────────
# TEMPORAL BUFFER
# ─────────────────────────────────────────────────────────────

pose_buffer = deque(maxlen=SEQ_LEN)

prev_pose = None

# ─────────────────────────────────────────────────────────────
# PREPROCESS FACE
# ─────────────────────────────────────────────────────────────

def preprocess_face(face_img):

    face_img = cv2.resize(face_img, (112, 112))

    face_img = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)

    face_img = face_img.astype(np.float32) / 255.0

    face_img = np.transpose(face_img, (2, 0, 1))

    face_img = np.expand_dims(face_img, axis=0)

    return face_img.astype(np.float32)

# ─────────────────────────────────────────────────────────────
# EXTRACT HEAD FEATURES
# ─────────────────────────────────────────────────────────────

def extract_head_features(landmarks):

    nose = landmarks[1]

    left_eye = landmarks[33]
    right_eye = landmarks[263]

    chin = landmarks[152]

    # approximate head pose features

    yaw = nose.x - 0.5

    pitch = nose.y - 0.5

    roll = right_eye.y - left_eye.y

    pose = np.array([pitch, yaw, roll], dtype=np.float32)

    return pose

# ─────────────────────────────────────────────────────────────
# HEAD MOVEMENT PREDICTION
# ─────────────────────────────────────────────────────────────

def predict_head_movement():

    if len(pose_buffer) < SEQ_LEN:
        return None, 0.0

    seq = np.array(pose_buffer, dtype=np.float32)

    seq = np.expand_dims(seq, axis=0)

    logits = hm_session.run(
        ['hm_logits'],
        {'pose_seq': seq}
    )[0]

    probs = softmax(logits[0])

    pred_idx = int(np.argmax(probs))

    return HEAD_CLASSES[pred_idx], float(probs[pred_idx])

# ─────────────────────────────────────────────────────────────
# EXPRESSION PREDICTION
# ─────────────────────────────────────────────────────────────

def predict_expression(face_crop):

    inp = preprocess_face(face_crop)

    logits = expr_session.run(
        ['expr_logits'],
        {'face_image': inp}
    )[0]

    probs = softmax(logits[0])

    pred_idx = int(np.argmax(probs))

    return EXPR_CLASSES[pred_idx], float(probs[pred_idx])

# ─────────────────────────────────────────────────────────────
# SOFTMAX
# ─────────────────────────────────────────────────────────────

def softmax(x):

    e = np.exp(x - np.max(x))

    return e / e.sum()

# ─────────────────────────────────────────────────────────────
# WEBCAM
# ─────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError('Could not open webcam')

print('\nPress Q to quit\n')

# ─────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = face_landmarker.detect(mp_image)

    expr_text = 'Expression: ---'
    hm_text = 'Head Movement: ---'

    if len(result.face_landmarks) > 0:

        landmarks = result.face_landmarks[0]

        # ─────────────────────────────────────────────
        # FACE BBOX
        # ─────────────────────────────────────────────

        xs = [lm.x for lm in landmarks]
        ys = [lm.y for lm in landmarks]

        h, w = frame.shape[:2]

        x1 = int(min(xs) * w)
        y1 = int(min(ys) * h)

        x2 = int(max(xs) * w)
        y2 = int(max(ys) * h)

        pad = 20

        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)

        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)

        face_crop = frame[y1:y2, x1:x2]

        # ─────────────────────────────────────────────
        # EXPRESSION
        # ─────────────────────────────────────────────

        if face_crop.size > 0:

            expr_label, expr_conf = predict_expression(face_crop)

            expr_text = f'Expression: {expr_label} ({expr_conf:.2f})'

        # ─────────────────────────────────────────────
        # HEAD FEATURES
        # ─────────────────────────────────────────────

        pose = extract_head_features(landmarks)

        pose_buffer.append(pose)

        hm_label, hm_conf = predict_head_movement()

        if hm_label is not None:
            hm_text = f'Head: {hm_label} ({hm_conf:.2f})'

        # ─────────────────────────────────────────────
        # DRAW BBOX
        # ─────────────────────────────────────────────

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

    # ─────────────────────────────────────────────────
    # DRAW TEXT
    # ─────────────────────────────────────────────────

    cv2.putText(
        frame,
        expr_text,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        hm_text,
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.imshow('ISL NMF Webcam Inference', frame)

    key = cv2.waitKey(1)

    if key & 0xFF == ord('q'):
        break

# ─────────────────────────────────────────────────────────────
# CLEANUP
# ─────────────────────────────────────────────────────────────

cap.release()

cv2.destroyAllWindows()



✓ ONNX models loaded


W0000 00:00:1778490852.343659 1845008 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1778490852.346997 1845008 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
W0000 00:00:1778490852.353068 1845010 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778490852.362312 1845011 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


✓ MediaPipe loaded

Press Q to quit

